In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import zipfile

zip_path = "/content/drive/MyDrive/archive (2).zip"
extract_path = "/content/movies_data"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Files extracted successfully!")

Files extracted successfully!


In [6]:
import os
os.listdir("/content/movies_data")

['movies.csv', 'links.csv', 'README.txt', 'ratings.csv', 'tags.csv']

In [8]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ---------------- LOAD DATA ----------------
movies = pd.read_csv("/content/movies_data/movies.csv")

# Keep needed columns
movies = movies[["movieId", "title", "genres"]]

# Replace separators in genres
movies["genres"] = movies["genres"].str.replace("|", " ", regex=False)

# ---------------- VECTORIZE ----------------
cv = CountVectorizer(stop_words="english")
matrix = cv.fit_transform(movies["genres"])

similarity = cosine_similarity(matrix)

# ---------------- UI ----------------
title = widgets.HTML(
    value="<h2 style='color:red;'>🎬 Movie Recommendation System</h2>"
)

search_box = widgets.Text(
    placeholder="Type movie name...",
    description="Search:",
    layout=widgets.Layout(width="500px"),
    style={'description_width': 'initial'}
)

movie_dropdown = widgets.Dropdown(
    options=list(movies["title"].head(50)),
    description="Movie:",
    layout=widgets.Layout(width="500px"),
    style={'description_width': 'initial'}
)

button = widgets.Button(
    description="Recommend",
    button_style="danger",
    icon="film"
)

output = widgets.Output()

# ---------------- SEARCH FILTER ----------------
def update_movies(change):
    text = search_box.value.lower()
    filtered = movies[movies["title"].str.lower().str.contains(text, na=False)]["title"].tolist()

    if filtered:
        movie_dropdown.options = filtered[:50]
    else:
        movie_dropdown.options = list(movies["title"].head(50))

search_box.observe(update_movies, names="value")

# ---------------- RECOMMEND FUNCTION ----------------
def recommend(movie_name):
    idx = movies[movies["title"] == movie_name].index[0]
    distances = list(enumerate(similarity[idx]))
    sorted_movies = sorted(distances, key=lambda x: x[1], reverse=True)[1:6]

    recs = []
    for i in sorted_movies:
        recs.append(movies.iloc[i[0]].title)
    return recs

def on_click(b):
    with output:
        clear_output()
        selected_movie = movie_dropdown.value

        print("Selected Movie:", selected_movie)
        print("\n🎯 Recommended Movies:\n")

        recommendations = recommend(selected_movie)

        for i, movie in enumerate(recommendations, start=1):
            print(f"{i}. {movie}")

button.on_click(on_click)

# ---------------- DISPLAY ----------------
display(title, search_box, movie_dropdown, button, output)

HTML(value="<h2 style='color:red;'>🎬 Movie Recommendation System</h2>")

Text(value='', description='Search:', layout=Layout(width='500px'), placeholder='Type movie name...', style=De…

Dropdown(description='Movie:', layout=Layout(width='500px'), options=('Toy Story (1995)', 'Jumanji (1995)', 'G…

Button(button_style='danger', description='Recommend', icon='film', style=ButtonStyle())

Output()